# 14 — Panel Data and Fixed Effects for Business Metrics
**References:** Wooldridge (2010) *Econometric Analysis of Cross Section and Panel Data* (2nd ed.), Ch. 10 · Angrist & Pischke (2009) *Mostly Harmless Econometrics*, Ch. 5 · Cameron & Miller (2015, *J. Human Resources*) "A Practitioner's Guide to Cluster-Robust Inference"

**Prerequisites:** causal_inference_course/04_confounding_bad_controls.ipynb (what counts as a
confounder vs. a bad control); statistics_course regression fundamentals.
**Connects to:** 11_did_business.ipynb and causal_inference_course/09_difference_in_differences.ipynb
(TWFE is a fixed-effects regression — this notebook derives the estimator those notebooks use as
a black box); 15_event_study.ipynb (built on the same panel structure).

## Narrative thread
```
Business panels: many stores/markets/customers observed repeatedly over time
   -> Entity fixed effects absorb time-invariant confounding -> the within estimator
   -> Time fixed effects absorb shocks common to everyone in a period
   -> When FE removes bias, and when it doesn't (time-varying confounders)
   -> statsmodels dummy-variable regression vs. linearmodels PanelOLS
   -> Clustered standard errors
```

## Why this notebook exists

This is a genuinely new topic relative to the rest of the causal inference material: **panel
data with fixed effects** is the workhorse tool behind almost every quasi-experimental design
covered so far (DiD's TWFE regression, in particular), but it deserves its own treatment because
it is used constantly on its own — not just as a DiD ingredient — whenever a business has
repeated observations of the same entities over time (stores, markets, customers, sales reps)
and wants to control for **stable, unobserved** differences across those entities without
needing an experiment or an event to compare before/after.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(14)

## Business setting: does local ad spend drive store revenue?

A retail chain runs 60 stores over 36 months. Each store has a fixed, unobserved "quality"
(location desirability, staff tenure, local brand reputation) that never changes over the
panel and correlates with **both** how much local marketing budget a store manager requests
and how much revenue the store generates — a classic omitted-variable-bias setup if we ignore
it.

In [2]:
# ── Simulate a store-month panel with a store-level confounder ──
n_stores = 60
n_months = 36
store_quality = np.random.normal(0, 1, n_stores)     # time-invariant, unobserved to a naive analyst
month_shock = np.random.normal(0, 0.5, n_months)     # common macro/seasonal shocks each month

rows = []
true_beta = 2.5  # true causal effect: $1k extra ad spend -> $2.5k extra revenue
for s in range(n_stores):
    # better-quality stores request more ad budget (manager confidence, bigger budgets) -> confounding
    base_ad_spend = 10 + 4 * store_quality[s]
    for t in range(n_months):
        ad_spend = max(0.5, base_ad_spend + np.random.normal(0, 1.5))
        revenue = (
            50 + 8 * store_quality[s] + 1.2 * month_shock[t]
            + true_beta * ad_spend + np.random.normal(0, 3)
        )
        rows.append((s, t, ad_spend, revenue))

panel = pd.DataFrame(rows, columns=["store", "month", "ad_spend", "revenue"])
panel.head()

,store,month,ad_spend,revenue
0,0,0,18.432510,106.370897
1,0,1,15.533028,92.790413
2,0,2,15.615198,99.614086
3,0,3,16.147445,104.308382
4,0,4,16.469629,100.073948


In [3]:
# ── Naive pooled OLS: ignores store_quality entirely ──
pooled = smf.ols("revenue ~ ad_spend", data=panel).fit()
print(f"Pooled OLS beta_hat: {pooled.params['ad_spend']:.3f}  (true = {true_beta})")
print("Biased upward: high-quality stores both spend more on ads and earn more revenue regardless.")

Pooled OLS beta_hat: 4.259  (true = 2.5)
Biased upward: high-quality stores both spend more on ads and earn more revenue regardless.


## The within (fixed-effects) estimator

The fixed-effects model adds a dummy for every entity:

$$Y_{it} = \alpha_i + \beta X_{it} + \varepsilon_{it}$$

Wooldridge (2010, Ch. 10) shows this is algebraically equivalent to the **within transformation**
— demean every variable at the entity level and run OLS on the demeaned data:

$$\ddot Y_{it} = Y_{it} - \bar Y_i, \qquad \ddot X_{it} = X_{it} - \bar X_i, \qquad
\ddot Y_{it} = \beta \ddot X_{it} + \ddot\varepsilon_{it}$$

Demeaning subtracts out anything **constant within an entity over the whole panel** —
`store_quality` never varies within a store, so it vanishes from $\ddot X$ and $\ddot Y$
entirely. This is why FE removes the bias here: the confounder is time-invariant.

In [4]:
# ── Manual within (demeaning) estimator ──
panel["revenue_dm"] = panel["revenue"] - panel.groupby("store")["revenue"].transform("mean")
panel["ad_spend_dm"] = panel["ad_spend"] - panel.groupby("store")["ad_spend"].transform("mean")

within = smf.ols("revenue_dm ~ ad_spend_dm - 1", data=panel).fit()
print(f"Manual within estimator beta_hat: {within.params['ad_spend_dm']:.3f}  (true = {true_beta})")

# ── Equivalent: least-squares dummy variable (LSDV) regression with store fixed effects ──
lsdv = smf.ols("revenue ~ ad_spend + C(store)", data=panel).fit()
print(f"LSDV (dummy-variable) beta_hat:     {lsdv.params['ad_spend']:.3f}  (true = {true_beta})")
print("The two match up to numerical precision -- this is the algebraic equivalence Wooldridge proves.")

Manual within estimator beta_hat: 2.522  (true = 2.5)
LSDV (dummy-variable) beta_hat:     2.522  (true = 2.5)
The two match up to numerical precision -- this is the algebraic equivalence Wooldridge proves.


## Adding time fixed effects

Month shocks (`month_shock`) are common to every store in a given month — a seasonal
promotion calendar, a shared input-cost shock. These do not bias `beta_hat` here because they
are uncorrelated with any store's *ad spend decision*, but leaving them in the residual
inflates the error variance and can bias standard errors if the shock also correlates with
`ad_spend` timing. Two-way fixed effects control for both simultaneously:

$$Y_{it} = \alpha_i + \lambda_t + \beta X_{it} + \varepsilon_{it}$$

In [5]:
twfe = smf.ols("revenue ~ ad_spend + C(store) + C(month)", data=panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["store"]}
)
print(f"Two-way FE beta_hat: {twfe.params['ad_spend']:.3f}  (true = {true_beta})")
print(f"Cluster-robust SE (by store): {twfe.bse['ad_spend']:.4f}")

Two-way FE beta_hat: 2.524  (true = 2.5)
Cluster-robust SE (by store): 0.0455


## The same model with linearmodels.PanelOLS

`linearmodels.panel.PanelOLS` is the standard library for panel regressions in Python: it
takes a `MultiIndex`-ed DataFrame (entity, time), supports entity/time effects natively, and
implements proper cluster-robust and Driscoll-Kraay covariance estimators without manually
constructing dummy columns.

In [6]:
from linearmodels.panel import PanelOLS

panel_idx = panel.set_index(["store", "month"])
mod = PanelOLS.from_formula(
    "revenue ~ ad_spend + EntityEffects + TimeEffects", data=panel_idx
)
res = mod.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                revenue   R-squared:                        0.6187
Estimator:                   PanelOLS   R-squared (Between):              0.5673
No. Observations:                2160   R-squared (Within):               0.6057
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.5675
Time:                        15:55:11   Log-likelihood                   -5400.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      3349.3
Entities:                          60   P-value                           0.0000
Avg Obs:                       36.000   Distribution:                  F(1,2064)
Min Obs:                       36.000                                           
Max Obs:                       36.000   F-statistic (robust):             3124.7
                            

## When fixed effects remove bias, and when they don't

Fixed effects only absorb confounding that is **constant within the entity across the whole
panel**. Two failure modes matter for business panels:

1. **Time-varying confounders.** If store quality *itself* were slowly improving (say, staff
   experience accumulating over the 36 months) and that improvement also drove both ad-budget
   requests and revenue, FE would not remove that bias — only the time-invariant *level* of
   quality is absorbed, not a trending component. The fix is to add controls for the
   time-varying confounder directly, or a store-specific linear trend.
2. **Reverse causality / simultaneity.** If managers *raise* ad spend *because* revenue dipped
   last month (a feedback loop), $X_{it}$ is correlated with $\varepsilon_{it}$ even after
   demeaning — FE does not fix simultaneity, only time-invariant omitted variables. That calls
   for an instrument (`17_instrumental_variables_business.ipynb`) or a lag structure with its
   own assumptions.

Let's demonstrate failure mode 1 directly: give store quality a **trend** instead of a level,
and watch FE fail to fully de-bias the estimate.

In [7]:
# ── Failure mode: a *trending* unobserved confounder that FE cannot remove ──
rows2 = []
for s in range(n_stores):
    base_ad_spend = 10 + 2 * store_quality[s]
    for t in range(n_months):
        trending_quality = store_quality[s] * (1 + 0.15 * t)  # quality gap widens sharply over time
        ad_spend = max(0.5, base_ad_spend + 1.2 * trending_quality + np.random.normal(0, 1.5))
        revenue = 50 + 8 * trending_quality + true_beta * ad_spend + np.random.normal(0, 3)
        rows2.append((s, t, ad_spend, revenue))

panel2 = pd.DataFrame(rows2, columns=["store", "month", "ad_spend", "revenue"])
twfe2 = smf.ols("revenue ~ ad_spend + C(store) + C(month)", data=panel2).fit(
    cov_type="cluster", cov_kwds={"groups": panel2["store"]}
)
print(f"Two-way FE beta_hat with a TRENDING confounder: {twfe2.params['ad_spend']:.3f}  (true = {true_beta})")
print("Still biased: FE only ever removes a time-invariant level, never a store-specific trend.")

# fix: add a store-specific linear time trend
panel2["store_trend"] = panel2["store"].astype(str) + "_t"
twfe2_trend = smf.ols(
    "revenue ~ ad_spend + C(store) + C(month) + C(store):month", data=panel2
).fit(cov_type="cluster", cov_kwds={"groups": panel2["store"]})
print(f"Two-way FE + store-specific trends beta_hat:   {twfe2_trend.params['ad_spend']:.3f}  (true = {true_beta})")

Two-way FE beta_hat with a TRENDING confounder: 6.511  (true = 2.5)
Still biased: FE only ever removes a time-invariant level, never a store-specific trend.
Two-way FE + store-specific trends beta_hat:   2.454  (true = 2.5)


## Clustered standard errors

Panel data violates the i.i.d. error assumption in an obvious way: observations from the same
store across months share unobserved shocks (autocorrelation), so plain OLS standard errors
are typically far too small. Cameron & Miller (2015) recommend **clustering at the level of
the treatment assignment / entity** — here, by store — which both `statsmodels`
(`cov_type="cluster"`) and `linearmodels` (`cov_type="clustered", cluster_entity=True`)
implement directly, as used above. With relatively few clusters (well under ~40-50), consider
a finite-sample correction (e.g., wild cluster bootstrap) rather than trusting the asymptotic
cluster-robust formula outright.

## Key takeaways

| Concept | Statement |
|---|---|
| Within estimator | Demeaning by entity removes any time-invariant confounder |
| LSDV equivalence | Entity dummies in a pooled regression give numerically identical estimates |
| Two-way FE | Adds time dummies to also absorb shocks common to all entities in a period |
| FE does NOT fix | Time-varying confounders, trends, or reverse causality/simultaneity |
| Clustered SEs | Cluster at the entity level; panel errors are not i.i.d. across time within an entity |

## References

| Author(s) | Title | Role here |
|---|---|---|
| Wooldridge (2010) | *Econometric Analysis of Cross Section and Panel Data*, Ch. 10 | Within estimator, LSDV equivalence |
| Angrist & Pischke (2009) | *Mostly Harmless Econometrics*, Ch. 5 | Fixed effects as a causal design |
| Cameron & Miller (2015, *J. Human Resources*) | "A Practitioner's Guide to Cluster-Robust Inference" | Clustered SEs |
